# Recursive guarded world model
GPU runtime. First run the smoke test; the ARC pilot downloads released weights and measures the actual WARC model. Paper numbers are reference-only, not the replay metric.


In [ ]:
import os, torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
print(torch.cuda.get_device_name(0), torch.__version__)
!git clone --depth 1 --branch research/recursive-world-model https://github.com/shekhawatxyz/fastai-2022 /content/recursive-research
%cd /content/recursive-research/experiments/recursive_world


In [ ]:
!python recursive_world.py --suite smoke --epochs 4 --seeds 1 --profile --out smoke


In [ ]:
%pip install -q datasets einops timm huggingface_hub arc-agi==0.9.9 arcengine==0.9.3
!git clone https://github.com/third-intelligence/WARC /content/WARC
!git -C /content/WARC checkout f0081ef00e867f63a08b1e4e7350d2baa0440267
!python recursive_world.py --suite arc --games ls20 --sessions 1 --limit 24 --seeds 1 --arms direct,copy,small,stale,grow --hard-arc --profile --out arc_pilot


In [ ]:
import pandas as pd
from pathlib import Path
p = Path('arc_pilot')
scores = pd.read_csv(p/'scores.csv')
display(scores.groupby('arm')[['changed_acc','false_edit','exact','change_f1','update_s']].mean())
scores.groupby(['step','arm']).changed_acc.mean().unstack().rolling(4,min_periods=1).mean().plot(figsize=(11,4),ylim=(0,1),title='Prequential replay (not MechEval)')
drift = pd.read_csv(p/'weight_drift.csv')
drift[drift.arm=='grow'].pivot_table(index='epoch',columns='parameter',values='relative_init').plot(legend=False,title='Grow: per-tensor weight drift')
display(pd.read_csv(p/'vocabulary.csv').query('arm == "grow" and active == True').tail(16))
print((p/'profile.txt').read_text())


For the full ablation and ensemble: run `python recursive_world.py --suite synthetic --epochs 12 --seeds 3 --ensemble --profile`. For larger ARC evaluation increase games, sessions and limit; do not tune on held-out games.
